In [1]:
!nvidia-smi

Tue Aug 25 06:58:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   70C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q -U transformers peft trl datasets accelerate
!pip uninstall -q -y torchao

In [3]:
from google.colab import drive
drive.mount("/content/drive")

from datasets import load_dataset

DATA_DIR = "/content/drive/MyDrive/ticket-triage"
data = load_dataset("json", data_files={
    "train": f"{DATA_DIR}/train.jsonl",
    "validation": f"{DATA_DIR}/val.jsonl",
})
data

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 5130
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 570
    })
})

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [5]:
SYSTEM_PROMPT = (
    "You are a triage assistant for an airline's customer support. "
    "Classify the customer tweet. Respond with json only, in exactly this format: "
    '{"intent": "...", "urgency": "...", "abusive": true/false}. '
    "intent must be one of: delay_disruption, checkin_boarding_issue, "
    "flight_cancellation_rebooking, lost_luggage, special_assistance, "
    "general_complaint, general_question, praise_feedback, spam_irrelevant, "
    "other_unclear. urgency must be one of: high, medium, low."
)


def triage(text):
  messages = [
      {"role": "system", "content": SYSTEM_PROMPT},
      {"role": "user", "content": text},
  ]
  prompt = tokenizer.apply_chat_template(
      messages, tokenize=False, add_generation_prompt=True
  )
  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
  with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=60, do_sample=False)
  return tokenizer.decode(
      out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
  )

tests = [
  "my bags are lost and I have no way to start getting them rerouted",
  "Thanks for the reply. We've got it sorted now.",
  "hello can i get a free flight to london rn",
]
for t in tests:
  print(t[:50], "->", triage(t))

my bags are lost and I have no way to start gettin -> {"intent": "general_question", "urgency": "medium", "abusive": false}
Thanks for the reply. We've got it sorted now. -> {"intent": "general_question", "urgency": "low", "abusive": false}
hello can i get a free flight to london rn -> {"intent": "general_question", "urgency": "low", "abusive": false}


In [6]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
)

In [7]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir="qwen-triage-lora",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="no",
    fp16=True,
    max_length=512,
    report_to="none",
)


trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=data["train"],
    eval_dataset=data["validation"],
    peft_config=peft_config,
    processing_class=tokenizer,
)

Tokenizing train dataset:   0%|          | 0/5130 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/5130 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5130 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/5130 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/570 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/570 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/570 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/570 [00:00<?, ? examples/s]

In [8]:
trainer.train()
trainer.save_model(f"{DATA_DIR}/qwen-triage-lora")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,0.683162,0.655613,0.647862,270181.000000,0.882231
200,0.627051,0.642861,0.657825,540683.000000,0.884002
300,0.628500,0.636915,0.638813,810378.000000,0.884074
400,0.582489,0.635146,0.592343,1079550.000000,0.884942
500,0.578499,0.631875,0.601388,1349494.000000,0.885162
600,0.578183,0.631280,0.600148,1619917.000000,0.885240
642,0.582790,0.631264,0.599735,1732416.000000,0.885232


In [9]:
model = trainer.model
model.eval()

for t in tests:
  print(t[:50], "->", triage(t))

my bags are lost and I have no way to start gettin -> {"intent": "lost_luggage", "urgency": "medium", "abusive": false}
Thanks for the reply. We've got it sorted now. -> {"intent": "praise_feedback", "urgency": "low", "abusive": false}
hello can i get a free flight to london rn -> {"intent": "general_question", "urgency": "low", "abusive": false}


In [11]:
import json
import pandas as pd

gold = pd.read_csv(f"{DATA_DIR}/gold_final.csv")

preds = []
broken = 0
for i,text in enumerate(gold["text_clean"], start=1):
  raw = triage(text)
  try:
    preds.append(json.loads(raw))
  except json.JSONDecpdeError:
    preds.append({"intent": None, "urgency": None, "abusive": None})
    broken += 1
    print("BROKEN JSON:", raw[:80])
  if i % 25 == 0:
    print(f"{i}/{len(gold)}")

print(f"done - broken JSON: {broken}/{len(gold)}")

pred_df = pd.DataFrame(preds, index=gold.index).add_suffix("_pred")
gold = pd.concat([gold, pred_df], axis=1)
gold.to_csv(f"{DATA_DIR}/gold_with_preds.csv", index=False)

25/300
50/300
75/300
100/300
125/300
150/300
175/300
200/300
225/300
250/300
275/300
300/300
done - broken JSON: 0/300


In [13]:
from sklearn.metrics import accuracy_score, classification_report, f1_score

valid = gold["intent_pred"].notna()
print(f"έγκυρα JSON: {valid.sum()}/{len(gold)}\n")

y_true = gold.loc[valid, "intent_human"]
y_pred = gold.loc[valid, "intent_pred"]

print("intent accuracy:", round(accuracy_score(y_true, y_pred), 3))
print("intent macro-F1:", round(f1_score(y_true, y_pred, average="macro"), 3))
print()
print(classification_report(y_true, y_pred, zero_division=0))


urg_acc = accuracy_score(gold.loc[valid, "urgency_human"], gold.loc[valid, "urgency_pred"])
ab_acc = accuracy_score(gold.loc[valid, "abusive_human"].astype(bool),
                        gold.loc[valid, "abusive_pred"].astype(bool))
print("urgency accuracy:", round(urg_acc, 3))
print("abusive accuracy:", round(ab_acc, 3))

έγκυρα JSON: 300/300

intent accuracy: 0.73
intent macro-F1: 0.645

                               precision    recall  f1-score   support

       checkin_boarding_issue       0.56      0.45      0.50        11
             delay_disruption       0.82      0.84      0.83        43
flight_cancellation_rebooking       0.46      0.60      0.52        10
            general_complaint       0.73      0.74      0.74        78
             general_question       0.70      0.59      0.64        44
                 lost_luggage       0.77      0.83      0.80        12
                other_unclear       0.63      0.70      0.67        37
              praise_feedback       0.88      0.94      0.91        47
              spam_irrelevant       0.58      0.54      0.56        13
           special_assistance       0.50      0.20      0.29         5

                     accuracy                           0.73       300
                    macro avg       0.66      0.64      0.64       300
       